In [7]:
import random

vocabulary = {
    "水果": {
        "apple": "蘋果",
        "banana": "香蕉",
    },
    "動物": {
        "cat": "貓咪",
    },
}

quiz_records = {"correct": [], "incorrect": []}


def all_words():
    """回傳所有單字，格式為 [(英文, 中文, 分類), ...]。"""
    return [
        (english, chinese, category)
        for category, words in vocabulary.items()
        for english, chinese in words.items()
    ]


def show_summary():
    """以文字格式顯示單字總數、分類與答題紀錄。"""
    words = all_words()
    print("=" * 36)
    print("             VocaBuddy 單字總覽")
    print("=" * 36)
    print(f"單字總數：{len(words)}")
    print("\n單字分類：")
    for category, category_words in vocabulary.items():
        print(f"  {category}（{len(category_words)} 個）：")
        print("    " + ", ".join(category_words.keys()))
    print("\n答題紀錄：")
    print(f"  答對：{len(quiz_records['correct'])} 題")
    print(f"  答錯：{len(quiz_records['incorrect'])} 題")
    print("=" * 36)


def run_quiz(question_count=3, answer_provider=input, category=None):
    """進行簡單中文翻英文測驗。"""
    candidates = [word for word in all_words() if category is None or word[2] == category]
    if not candidates:
        print("找不到可測驗的單字。")
        return
    questions = random.sample(candidates, min(question_count, len(candidates)))
    correct_count = 0
    for number, (english, chinese, word_category) in enumerate(questions, start=1):
        answer = answer_provider(f"第 {number} 題 [{word_category}] {chinese} 的英文是：").strip().lower()
        if answer == english.lower():
            print("答對！")
            quiz_records["correct"].append(english)
            correct_count += 1
        else:
            print(f"答錯，正確答案是：{english}")
            quiz_records["incorrect"].append({"question": chinese, "answer": answer, "correct": english})
    print(f"本次成績：{correct_count}/{len(questions)} 題答對")


def add_word(english, chinese, category):
    """新增單字到指定分類。"""
    vocabulary.setdefault(category, {})[english.strip().lower()] = chinese.strip()


In [8]:
import json
from IPython.display import HTML, Javascript, clear_output, display

words_for_ui = [
    {"english": english, "chinese": chinese, "category": category}
    for category, words in vocabulary.items()
    for english, chinese in words.items()
]
words_json = json.dumps(words_for_ui, ensure_ascii=False)
initial_word = words_for_ui[0] if words_for_ui else {"english": "目前沒有單字", "chinese": "請先新增單字", "category": "尚未分類"}
category_options = "".join(f"<option value='{category}'>{category}</option>" for category in ["全部分類"] + list(vocabulary.keys()))
library_html = "".join(
    f"<div class='vb-category'><div class='vb-category-title'>{category} <span class='vb-muted'>({len(category_words)} 個)</span></div>"
    + "".join(f"<span class='vb-word'>{english}</span>" for english in category_words)
    + "</div>"
    for category, category_words in vocabulary.items()
)

html_ui = """
<!DOCTYPE html>
<html lang="zh-Hant">
<head>
<meta charset="UTF-8">
<style>
.vocabuddy-ui { --ink:#173b50; --muted:#617984; --teal:#087f73; --mint:#e8f7f3; --cream:#fff8ed; --line:#d9e7e7; max-width:940px; margin:0 auto; padding:12px 8px 28px; color:var(--ink); font-family:"Trebuchet MS","Segoe UI",sans-serif; line-height:1.5; }
.vocabuddy-ui *, .vocabuddy-ui *::before, .vocabuddy-ui *::after { box-sizing:border-box; }
.vb-hero { padding:30px 34px; border:1px solid #c8e5df; border-radius:20px; background:linear-gradient(135deg,var(--mint),var(--cream)); }
.vb-eyebrow { margin:0 0 8px; color:var(--teal); font-size:13px; font-weight:700; letter-spacing:.08em; }
.vb-hero h1 { margin:0 0 6px; font-size:clamp(28px,5vw,40px); line-height:1.15; }
.vb-hero p { margin:0; color:var(--muted); font-size:16px; }
.vb-stats { display:grid; grid-template-columns:repeat(3,1fr); gap:12px; margin:16px 0; }
.vb-stat { padding:18px; min-height:88px; border:1px solid var(--line); border-radius:14px; background:#fff; }
.vb-stat strong { display:block; color:var(--teal); font-size:28px; line-height:1.2; }
.vb-stat span, .vb-muted { color:var(--muted); font-size:13px; }
.vb-section { margin-top:16px; padding:22px; border:1px solid var(--line); border-radius:16px; background:#fff; }
.vb-tabs { display:flex; gap:8px; margin-bottom:16px; overflow-x:auto; }
.vb-tab, .vb-button { min-height:44px; border:1px solid transparent; border-radius:9px; padding:10px 16px; font:inherit; font-weight:700; cursor:pointer; transition:background .2s,transform .2s; }
.vb-tab { background:#f1f6f5; color:var(--muted); white-space:nowrap; }
.vb-tab.active, .vb-tab:hover { background:var(--ink); color:#fff; }
.vb-button { background:var(--teal); color:#fff; }
.vb-button.secondary { background:#fff; color:var(--ink); border-color:var(--line); }
.vb-button:disabled { cursor:not-allowed; opacity:.48; }
.vb-button:focus-visible, .vb-tab:focus-visible, .vb-answer:focus-visible, .vb-select:focus-visible, .vb-flashcard:focus-visible { outline:3px solid #9bd8ce; outline-offset:2px; }
.vb-panel { display:none; }
.vb-panel.active { display:block; }
.vb-filter-row { display:flex; align-items:end; flex-wrap:wrap; gap:12px; margin-bottom:16px; }
.vb-field { display:flex; flex-direction:column; gap:5px; color:var(--muted); font-size:13px; font-weight:700; }
.vb-select, .vb-answer { min-height:44px; border:1px solid var(--line); border-radius:8px; padding:9px 12px; color:var(--ink); background:#fff; font:inherit; }
.vb-flashcard { display:flex; align-items:center; justify-content:center; min-height:210px; padding:28px; border:1px solid #b9ddd7; border-radius:16px; background:var(--ink); color:#fff; text-align:center; cursor:pointer; user-select:none; }
.vb-card-label { display:block; margin-bottom:8px; color:#bfe5de; font-size:13px; }
.vb-card-word { display:block; font-size:clamp(30px,7vw,52px); font-weight:700; }
.vb-card-hint { margin:10px 0 0; color:var(--muted); text-align:center; font-size:14px; }
.vb-actions { display:flex; justify-content:center; flex-wrap:wrap; gap:9px; margin-top:16px; }
.vb-category { margin-bottom:16px; }
.vb-category:last-child { margin-bottom:0; }
.vb-category-title { margin-bottom:7px; font-weight:700; }
.vb-word { display:inline-block; margin:0 5px 5px 0; padding:6px 10px; border-radius:7px; background:#edf7f5; color:#236e68; font-size:13px; }
.vb-question { min-height:144px; padding:24px; border-radius:14px; background:var(--ink); color:#fff; }
.vb-question small { display:block; margin-bottom:10px; color:#bfe5de; }
.vb-question strong { font-size:30px; }
.vb-answer-row { display:flex; gap:8px; margin-top:14px; }
.vb-answer { flex:1; min-width:0; }
.vb-feedback { min-height:24px; margin-top:12px; padding:10px 13px; border-radius:8px; color:var(--muted); }
.vb-feedback.good { background:#e4f6eb; color:#17633d; }
.vb-feedback.bad { background:#fff0ed; color:#a34437; }
.vb-progress { height:7px; margin:14px 0 0; overflow:hidden; border-radius:99px; background:#e4eeee; }
.vb-progress-bar { height:100%; width:0; border-radius:inherit; background:var(--teal); transition:width .2s ease; }
@media (max-width:650px) { .vb-stats{grid-template-columns:1fr;} .vb-hero{padding:24px;} .vb-section{padding:16px;} .vb-answer-row{flex-direction:column;} }
@media (prefers-reduced-motion:reduce) { .vb-tab,.vb-button,.vb-progress-bar{transition:none;} }
</style>
</head>
<body>
<div class="vocabuddy-ui">
  <header class="vb-hero"><p class="vb-eyebrow">DAILY WORD PRACTICE</p><h1>VocaBuddy</h1><p>翻一張卡、回答一題，讓單字練習變得簡單而有節奏。</p></header>
  <section class="vb-stats" aria-label="學習統計"><div class="vb-stat"><strong id="totalWords">__TOTAL_WORDS__</strong><span>單字總數</span></div><div class="vb-stat"><strong id="correctCount">0</strong><span>答對題數</span></div><div class="vb-stat"><strong id="accuracy">0%</strong><span>目前正確率</span></div></section>
  <section class="vb-section">
    <nav class="vb-tabs" aria-label="學習模式"><button class="vb-tab active" data-panel="cardsPanel" type="button">單字卡</button><button class="vb-tab" data-panel="quizPanel" type="button">單字測驗</button><button class="vb-tab" data-panel="libraryPanel" type="button">單字分類</button></nav>
    <div id="cardsPanel" class="vb-panel active"><div class="vb-filter-row"><label class="vb-field">分類<select id="cardCategory" class="vb-select" aria-label="選擇單字卡分類">__CATEGORY_OPTIONS__</select></label><span id="cardPosition" class="vb-muted">1 / __TOTAL_WORDS__</span></div><div id="flashcard" class="vb-flashcard" tabindex="0" role="button" aria-label="點擊翻開單字卡"><div><span id="cardLabel" class="vb-card-label">__INITIAL_CATEGORY__</span><span id="cardWord" class="vb-card-word">__INITIAL_ENGLISH__</span></div></div><p class="vb-card-hint">點擊卡片或按 Enter 翻面</p><div class="vb-actions"><button id="previousButton" class="vb-button secondary" type="button">上一張</button><button id="nextButton" class="vb-button" type="button">下一張</button><button id="randomButton" class="vb-button secondary" type="button">隨機單字</button></div></div>
    <div id="quizPanel" class="vb-panel"><div class="vb-filter-row"><label class="vb-field">測驗分類<select id="quizCategory" class="vb-select" aria-label="選擇測驗分類">__CATEGORY_OPTIONS__</select></label><label class="vb-field">題數<select id="quizLength" class="vb-select" aria-label="選擇測驗題數"><option value="3">3 題</option><option value="5">5 題</option><option value="10">10 題</option></select></label><button id="startQuizButton" class="vb-button" type="button">開始測驗</button></div><div id="quizContent" hidden><div id="quizQuestion" class="vb-question"><small>第 1 題</small><strong>準備開始</strong></div><div class="vb-progress" aria-label="測驗進度"><div id="quizProgress" class="vb-progress-bar"></div></div><div class="vb-answer-row"><input id="answerInput" class="vb-answer" type="text" placeholder="輸入英文答案" autocomplete="off" aria-label="英文答案"><button id="submitAnswerButton" class="vb-button" type="button">確認答案</button><button id="nextQuestionButton" class="vb-button secondary" type="button" disabled>下一題</button></div><div id="quizFeedback" class="vb-feedback" aria-live="polite">輸入答案後按確認答案。</div></div></div>
    <div id="libraryPanel" class="vb-panel"><div id="categoryList">__LIBRARY_HTML__</div></div>
  </section>
</div>
</body>
</html>
"""

html_ui = (html_ui.replace("__WORDS_JSON__", words_json)
    .replace("__TOTAL_WORDS__", str(len(words_for_ui)))
    .replace("__CATEGORY_OPTIONS__", category_options)
    .replace("__INITIAL_ENGLISH__", initial_word["english"])
    .replace("__INITIAL_CATEGORY__", initial_word["category"])
    .replace("__LIBRARY_HTML__", library_html))

js_ui = """
const root = element;
const WORDS = __WORDS_JSON__;
let cardIndex = 0, cardFlipped = false, filteredCards = [...WORDS];
let quizQuestions = [], quizIndex = 0, quizCorrect = 0, quizAnswered = false;
let records = { correct: 0, incorrect: 0 };
const $ = (id) => root.querySelector(`#${id}`);
const all = (selector) => root.querySelectorAll(selector);
const categories = ["全部分類", ...new Set(WORDS.map((word) => word.category))];
function updateStats() { const total = records.correct + records.incorrect; $("totalWords").textContent = WORDS.length; $("correctCount").textContent = records.correct; $("accuracy").textContent = `${total ? Math.round(records.correct / total * 100) : 0}%`; }
function renderCard() { const word = filteredCards[cardIndex]; if (!word) return; $("cardLabel").textContent = cardFlipped ? "中文意思" : word.category; $("cardWord").textContent = cardFlipped ? word.chinese : word.english; $("cardPosition").textContent = `${cardIndex + 1} / ${filteredCards.length}`; }
function flipCard() { cardFlipped = !cardFlipped; renderCard(); }
function moveCard(step) { cardIndex = (cardIndex + step + filteredCards.length) % filteredCards.length; cardFlipped = false; renderCard(); }
function randomCard() { cardIndex = Math.floor(Math.random() * filteredCards.length); cardFlipped = false; renderCard(); }
function startQuiz() { const category = $("quizCategory").value; const count = Number($("quizLength").value); const pool = WORDS.filter((word) => category === "全部分類" || word.category === category); quizQuestions = [...pool].sort(() => Math.random() - .5).slice(0, Math.min(count, pool.length)); quizIndex = 0; quizCorrect = 0; $("quizContent").hidden = false; $("startQuizButton").textContent = "重新開始"; showQuestion(); }
function showQuestion() { if (quizIndex >= quizQuestions.length) { $("quizQuestion").innerHTML = `<small>測驗完成</small><strong>本次答對 ${quizCorrect} / ${quizQuestions.length} 題</strong>`; $("quizFeedback").className = "vb-feedback good"; $("quizFeedback").textContent = "完成得很好，繼續累積你的單字量！"; $("submitAnswerButton").disabled = true; $("nextQuestionButton").disabled = true; $("quizProgress").style.width = "100%"; updateStats(); return; } const word = quizQuestions[quizIndex]; quizAnswered = false; $("quizQuestion").innerHTML = `<small>第 ${quizIndex + 1} / ${quizQuestions.length} 題　·　${word.category}</small><strong>${word.chinese}</strong>`; $("quizProgress").style.width = `${quizIndex / quizQuestions.length * 100}%`; $("answerInput").value = ""; $("answerInput").disabled = false; $("submitAnswerButton").disabled = false; $("nextQuestionButton").disabled = true; $("quizFeedback").className = "vb-feedback"; $("quizFeedback").textContent = "輸入英文答案後按確認答案。"; }
function submitAnswer() { if (quizAnswered || quizIndex >= quizQuestions.length) return; const word = quizQuestions[quizIndex]; const answer = $("answerInput").value.trim().toLowerCase(); quizAnswered = true; $("answerInput").disabled = true; $("submitAnswerButton").disabled = true; $("nextQuestionButton").disabled = false; if (answer === word.english.toLowerCase()) { quizCorrect++; records.correct++; $("quizFeedback").className = "vb-feedback good"; $("quizFeedback").textContent = "答對了！繼續保持這個節奏。"; } else { records.incorrect++; $("quizFeedback").className = "vb-feedback bad"; $("quizFeedback").innerHTML = `正確答案是 <b>${word.english}</b>，下一題繼續加油。`; } updateStats(); }
function switchPanel(event) { all(".vb-tab").forEach((tab) => tab.classList.remove("active")); all(".vb-panel").forEach((panel) => panel.classList.remove("active")); event.currentTarget.classList.add("active"); $(event.currentTarget.dataset.panel).classList.add("active"); }
all(".vb-tab").forEach((tab) => tab.addEventListener("click", switchPanel));
$("flashcard").addEventListener("click", flipCard);
$("flashcard").addEventListener("keydown", (event) => { if (event.key === "Enter" || event.key === " ") { event.preventDefault(); flipCard(); } });
$("previousButton").addEventListener("click", () => moveCard(-1)); $("nextButton").addEventListener("click", () => moveCard(1)); $("randomButton").addEventListener("click", randomCard);
$("cardCategory").addEventListener("change", (event) => { filteredCards = event.target.value === "全部分類" ? [...WORDS] : WORDS.filter((word) => word.category === event.target.value); cardIndex = 0; cardFlipped = false; renderCard(); });
$("startQuizButton").addEventListener("click", startQuiz); $("submitAnswerButton").addEventListener("click", submitAnswer); $("nextQuestionButton").addEventListener("click", () => { quizIndex++; showQuestion(); }); $("answerInput").addEventListener("keydown", (event) => { if (event.key === "Enter") submitAnswer(); });
updateStats(); renderCard();
""".replace("__WORDS_JSON__", words_json)

clear_output(wait=True)
display(HTML(html_ui))
display(Javascript(js_ui))


<IPython.core.display.Javascript object>